# Key Biodiversity Areas (KBA)

Processing the World Database of Key Biodiversity Areas (WDKBA). The dataset contains
a polygon layer (site boundaries) and a point layer (site centroids). The processing
includes cleaning both layers and converting them to mbtiles for upload to Mapbox.

**Source:** [keybiodiversityareas.org](https://www.keybiodiversityareas.org/kba-data/request)

**Citation:** KBA Secretariat (2024). World Database of Key Biodiversity Areas. Developed by the KBA Secretariat on behalf of the KBA Partnership. Available at [keybiodiversityareas.org](https://www.keybiodiversityareas.org).

## Setup

In [ ]:
import json
import logging
import os
import subprocess
import time
from pathlib import Path

import antimeridian
import boto3
import geopandas as gpd
import requests
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

path_raw = "../data/raw/KBAsGlobal_2025_March_01"
path_out = "../data/processed/kba"
os.makedirs(path_out, exist_ok=True)

In [ ]:
def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    opts: str,
):
    """Use tippecanoe to create mbtiles from a GeoJSON file."""
    cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
    logger.info(f"Running: {cmd}")
    r = subprocess.call(cmd, shell=True)
    if r != 0:
        raise RuntimeError(f"tippecanoe failed with exit code {r}")
    return r


def upload_mbtiles(mbtiles_path: str, tileset_id: str, name: str):
    """Upload an mbtiles file to Mapbox using the Uploads API."""
    token = os.environ["MAPBOX_ACCESS_TOKEN"]
    username = os.environ["MAPBOX_USERNAME"]
    base_url = f"https://api.mapbox.com/uploads/v1/{username}"
    full_tileset = f"{username}.{tileset_id}"

    file_size_mb = os.path.getsize(mbtiles_path) / (1024 * 1024)
    print(f"Uploading {mbtiles_path} ({file_size_mb:.1f} MB) \u2192 {full_tileset}")

    resp = requests.post(f"{base_url}/credentials", params={"access_token": token})
    resp.raise_for_status()
    creds = resp.json()

    s3 = boto3.client(
        "s3",
        aws_access_key_id=creds["accessKeyId"],
        aws_secret_access_key=creds["secretAccessKey"],
        aws_session_token=creds["sessionToken"],
        region_name="us-east-1",
    )
    s3.upload_file(mbtiles_path, creds["bucket"], creds["key"])

    resp = requests.post(
        base_url,
        params={"access_token": token},
        json={"url": creds["url"], "tileset": full_tileset, "name": name},
    )
    resp.raise_for_status()
    upload_id = resp.json()["id"]

    while True:
        time.sleep(5)
        resp = requests.get(f"{base_url}/{upload_id}", params={"access_token": token})
        resp.raise_for_status()
        status = resp.json()
        progress = status.get("progress", 0)
        print(f"  progress: {progress:.0%}", end="")
        if status.get("complete"):
            print(" \u2014 done!")
            return status
        if status.get("error"):
            print(f"\n  ERROR: {status['error']}")
            return status
        print()

## Load data

Update the filenames below to match your downloaded KBA shapefiles.

In [ ]:
kba_polygons = gpd.read_file(os.path.join(path_raw, "KBAsGlobal_2025_March_01_POL.shp"), engine="pyogrio")
kba_points = gpd.read_file(os.path.join(path_raw, "KBAsGlobal_2025_March_01_PNT.shp"), engine="pyogrio")

print(f"Polygons: {len(kba_polygons)} features, Points: {len(kba_points)} features")

## Clean data

In [ ]:
# Polygons
if kba_polygons.crs and kba_polygons.crs.to_epsg() != 4326:
    kba_polygons = kba_polygons.to_crs(epsg=4326)

invalid_count = (~kba_polygons.geometry.is_valid).sum()
if invalid_count > 0:
    kba_polygons["geometry"] = kba_polygons.geometry.make_valid()

kba_polygons = kba_polygons[~kba_polygons.geometry.is_empty & kba_polygons.geometry.notna()]
kba_polygons = kba_polygons.drop_duplicates(subset="SitRecID")

# Points
if kba_points.crs and kba_points.crs.to_epsg() != 4326:
    kba_points = kba_points.to_crs(epsg=4326)

kba_points = kba_points[~kba_points.geometry.is_empty & kba_points.geometry.notna()]
kba_points = kba_points.drop_duplicates(subset="SitRecID")

print(f"Clean polygons: {len(kba_polygons)}, Clean points: {len(kba_points)}")

## Export to GeoJSON and fix antimeridian

Polygons crossing the 180\u00b0/-180\u00b0 line get rendered as spanning the entire globe in
GeoJSON. We fix those features by splitting them at the antimeridian.

In [ ]:
polygons_geojson = os.path.join(path_out, "kba_polygons.geojson")
points_geojson = os.path.join(path_out, "kba_points.geojson")

kba_polygons.to_file(polygons_geojson, driver="GeoJSON")
kba_points.to_file(points_geojson, driver="GeoJSON")


def _crosses_antimeridian(feature):
    """Check if a feature's coordinates span the antimeridian."""
    def _extract_lons(coords):
        if isinstance(coords[0], (int, float)):
            return [coords[0]]
        lons = []
        for c in coords:
            lons.extend(_extract_lons(c))
        return lons

    lons = _extract_lons(feature["geometry"]["coordinates"])
    if not lons:
        return False
    return max(lons) - min(lons) > 180


with open(polygons_geojson) as f:
    geojson = json.load(f)

fixed_count = 0
for i, feature in enumerate(geojson["features"]):
    if _crosses_antimeridian(feature):
        geojson["features"][i] = antimeridian.fix_geojson(feature)
        fixed_count += 1

with open(polygons_geojson, "w") as f:
    json.dump(geojson, f)

print(f"Antimeridian: {fixed_count} features fixed, {len(geojson['features'])} total")

## Create mbtiles

Tippecanoe settings:
- `-Z1 -z10`: zoom levels 1 through 10
- `-B4`: base zoom 4 — all features guaranteed present at zoom 4+
- `--drop-densest-as-needed`: drop features at low zooms to stay under the 500KB tile size limit
- `--force`: overwrite existing mbtiles

In [ ]:
tippecanoe_polygon_opts = " ".join([
    "--force",
    "--read-parallel",
    "-Z1",
    "-z10",
    "-B4",
    "--drop-densest-as-needed",
    "--maximum-tile-bytes=1500000",
])

create_mbtiles(
    polygons_geojson,
    os.path.join(path_out, "kba_polygons.mbtiles"),
    "kba_polygons",
    tippecanoe_polygon_opts,
)

In [ ]:
tippecanoe_point_opts = " ".join([
    "--force",
    "--read-parallel",
    "-Z1",
    "-z10",
    "-B1",
    "--no-tile-size-limit",
    "--no-feature-limit",
])

create_mbtiles(
    points_geojson,
    os.path.join(path_out, "kba_points.mbtiles"),
    "kba_points",
    tippecanoe_point_opts,
)

## Upload to Mapbox

Requires `MAPBOX_ACCESS_TOKEN` (secret token with `uploads:read` and `uploads:write`
scopes) and `MAPBOX_USERNAME` in `../.env`.

Edit the `tileset_id` and `name` below before running.

In [ ]:
upload_mbtiles(
    mbtiles_path=os.path.join(path_out, "kba_polygons.mbtiles"),
    tileset_id="kba_polygons",       # \u2190 edit this
    name="KBA Polygons",             # \u2190 edit this
)

In [ ]:
upload_mbtiles(
    mbtiles_path=os.path.join(path_out, "kba_points.mbtiles"),
    tileset_id="kba_points",         # \u2190 edit this
    name="KBA Points",               # \u2190 edit this
)